In [5]:
import pandas as pd
import numpy as np
from sklearn.cross_decomposition import CCA

In [6]:
trainIDs =  ["F20208_heart_1Ses_time2slc_MskCrop128_V2_3D100epat2_L1_4ChTrans128fold0_prec32_pythaemodel-custom_factor_vae",
            "F20208_heart_1Ses_time2slc_MskCrop128_V2_3D100epat2_seed1993_L1_4ChTrans128fold0_prec32_pythaemodel-custom_factor_vae",
            "F20208_heart_1Ses_time2slc_MskCrop128_V2_3D100epat2_seed42_L1_4ChTrans128fold0_prec32_pythaemodel-custom_factor_vae",
            "F20208_heart_1Ses_time2slc_MskCrop128_V2_3D100ep_seed1994_L1_4ChTrans128fold0_prec32_pythaemodel-custom_factor_vae",
            "F20208_heart_1Ses_time2slc_MskCrop128_V2_3D100ep_seed2023_L1_4ChTrans128fold0_prec32_pythaemodel-custom_factor_vae",]

res_root = r"/project/ukbblatent/Out/Results"
gwas_folder = "GWAS_fullDSV2"

In [7]:
def pairwiseCCA(models, n_latents, max_iter=10000):
    n_models = len(models)
    
    for i in range(n_models):
        if type(models[i]) is pd.DataFrame:
            models[i] = models[i].to_numpy()
    
    cca = CCA(n_components=n_latents, max_iter=max_iter)
    
    print("Calculating pairwise CCAs...")
    final_latent = models[0]
    for i in range(1, len(models)):
        print(f"CCA between previous and model {i}")
        cca.fit(final_latent, models[i])
        X_c, Y_c = cca.transform(final_latent, models[i])
        final_latent = (X_c + Y_c) / 2.0

    return final_latent    

In [8]:
print("Reading the latents.....")
processed_latents = []
for trainID in trainIDs:
    df = pd.read_table(f"{res_root}/{trainID}/{gwas_folder}/processed_raw.tsv").sort_values("FID").reset_index(drop=True)
    IID = df.IID
    FID = df.FID
    df.drop(columns=["IID", "FID"], inplace=True)
    latents = df.columns.to_list()
    processed_latents.append(df)
    
final_components = pairwiseCCA(processed_latents, n_latents=len(latents))

merged_latents = pd.DataFrame(final_components, columns=latents)
merged_latents['IID'] = IID
merged_latents['FID'] = FID

cols = ['FID', 'IID'] + [col for col in merged_latents.columns if col not in ['FID', 'IID']]
merged_latents = merged_latents[cols]


Reading the latents.....
Calculating pairwise CCAs...
CCA between previous and model 1
CCA between previous and model 2
CCA between previous and model 3
CCA between previous and model 4


In [9]:
merged_latents.to_csv("/group/glastonbury/soumick/GWAS/merged_models/5Seeds_FVAE_128_Crop3D_DSV2_fold0/pairwiseCCA/merged_latents_raw.tsv", sep="\t", index=False)